# Did the One-Child Policy Actually Work?
### Two Synthetic Control Analyses of China's 1979 Fertility Policy

**Author:** Srisha Raj  
**Data:** World Bank World Development Indicators  
**Method:** Synthetic Control Method (Abadie, Diamond & Hainmueller, 2010)

---

## Research Question

China's fertility rate was already falling sharply before 1979, from 6.1 births per woman in 1960 to 2.8 by the time the One-Child Policy launched. **How much of China's fertility decline was caused by the policy, versus economic and demographic forces that were already in motion?**

This notebook investigates that question along **two outcomes**:

1. **Total Fertility Rate (TFR)**: did the policy decrease fertility below structural development trends?

2. **Sex Ratio at Birth**: did the policy change *how* families made fertility decisions, specifically by intensifying son preference under a one-child constraint?

A country's TFR can fall for many structural reasons, such as urbanization or rising incomes, which have nothing to do with government fertility policy. Sex ratio at birth, however, hold a biological baseline (~1.03–1.06 male births per female birth) almost everywhere, regardless of development. A sustained deviation from that baseline is a much more direct fingerprint of policy-driven behavior.

Running both **in tandem** through the same causal design lets us ask a sharper question than either alone: *did the policy change the pace of fertility decline, the nature of fertility decisions, both, or neither?*

### Notebook Structure
1. Setup
2. Exploratory Data Analysis
3. Motivating Analysis: Difference-in-Differences (DiD)
4. Synthetic Control — Outcome 1: Total Fertility Rate
5. Synthetic Control — Outcome 2: Sex Ratio at Birth
6. Placebo Tests 
7. Comparative Interpretation
8. Limitations & Next Steps


## 1. Setup & Data Loading

In [1]:
# pysyncon implements the Abadie et al. (2010) Synthetic Control Method
try:
    import pysyncon
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pysyncon', '-q',
                           '--break-system-packages'])
    import pysyncon

try:
    import statsmodels.formula.api as smf
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'statsmodels', '-q',
                           '--break-system-packages'])
    import statsmodels.formula.api as smf


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as plticker
import seaborn as sns
from pysyncon import Dataprep, Synth

# Plot styling
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})


### 1.1 Load Raw Data

Five World Bank indicators are used:

| Indicator | Code | Role |
|---|---|---|
| Total Fertility Rate | SP.DYN.TFRT.IN | Outcome 1 |
| Sex Ratio at Birth | SP.POP.BRTH.MF | Outcome 2 |
| GDP (current USD) | NY.GDP.MKTP.CD | Predictor — economic development |
| Urban population (% of total) | SP.URB.TOTL.IN.ZS | Predictor — structural modernization |
| Under-5 mortality rate | SH.DYN.MORT | Predictor — child survival dynamics |

**Note on country names:** The World Bank uses `Korea, Rep.` and `Turkiye` — standardized below.


In [3]:
# ── File paths ────────────────────────────────────────────────────────────
DATA = {
    'fertility' : 'data/fertility_rate.csv',
    'sex_ratio' : 'data/sex_ratio.csv',
    'gdp'       : 'data/gdp.csv',
    'urban'     : 'data/urban_population_pct.csv',
    'mortality' : 'data/child_mortality.csv',
}

# ── Country configuration ─────────────────────────────────────────────────
# Vietnam excluded: had its own government fertility campaigns from the late 1970s,
# which would contaminate the counterfactual.
DONOR_COUNTRIES = [
    'Thailand', 'Turkey', 'Pakistan', 'Philippines',
    'Sri Lanka', 'Morocco', 'Tunisia', 'South Korea',
    'Brazil', 'Mexico'
]
ALL_COUNTRIES = ['China'] + DONOR_COUNTRIES

NAME_MAP = {
    'Korea, Rep.'  : 'South Korea',
    'Turkiye'      : 'Turkey',
}

STR_YEARS = [str(y) for y in range(1960, 2016)]
PRE_YEARS  = list(range(1969, 1979))   # optimization window: 1969–1978
POLICY_YEAR = 1979

def load_wb(path, indicator_name=None):
    """Load a World Bank wide-format CSV and return a long-format DataFrame."""
    df = pd.read_csv(path, skiprows=4)
    if indicator_name:
        df = df[df['Indicator Name'] == indicator_name]
    df = df[['Country Name'] + STR_YEARS].copy()
    df['Country Name'] = df['Country Name'].replace(NAME_MAP)
    df = df[df['Country Name'].isin(ALL_COUNTRIES)]
    df = df.melt(id_vars='Country Name', var_name='Year', value_name='value')
    df['Year'] = df['Year'].astype(int)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df


In [6]:
fert  = load_wb(DATA['fertility']).rename(columns={'value': 'Fertility'})
sexr  = load_wb(DATA['sex_ratio']).rename(columns={'value': 'SexRatio'})
gdp   = load_wb(DATA['gdp']).rename(columns={'value': 'GDP'})
urban = load_wb(DATA['urban'],
                indicator_name='Urban population (% of total population)'
               ).rename(columns={'value': 'Urban_Pct'})
mort  = load_wb(DATA['mortality']).rename(columns={'value': 'Mortality'})

# Merge all indicators; log-transform GDP (right-skewed across donor pool)
df = (fert
      .merge(sexr,  on=['Country Name', 'Year'])
      .merge(gdp,   on=['Country Name', 'Year'])
      .merge(urban, on=['Country Name', 'Year'])
      .merge(mort,  on=['Country Name', 'Year']))

df['log_GDP'] = np.log(df['GDP'])
df = df.drop(columns='GDP')

# Restrict to 1969–2015
# Lower bound: 1969 is the first year China has complete mortality data
# Upper bound: 2015 avoids post-2015 data sparsity in some donors
df = df[(df['Year'] >= 1969) & (df['Year'] <= 2015)]

print(f"Shape: {df.shape}")
print(f"Countries: {sorted(df['Country Name'].unique())}")
print(f"Missing values:\n{df.isnull().sum()}")


FileNotFoundError: [Errno 2] No such file or directory: 'data/fertility_rate.csv'

---
## 2. Exploratory Data Analysis

Three questions, asked of both outcomes:
1. What did China's trajectory look like, and when did each outcome start diverging from its pre-1979 pattern?
2. How does the donor pool compare to China pre-1979?
3. Do the two outcomes tell visibly different stories even before any modeling?


In [ ]:
# ── Fig 1: China's trajectory — both outcomes side by side ────────────────
china = df[df['Country Name'] == 'China'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(china['Year'], china['Fertility'], color='#1B4F72', linewidth=2.5)
axes[0].axvline(1979, color='crimson', linestyle='--', linewidth=1.5, label='Policy Start (1979)')
axes[0].set_title("China: Total Fertility Rate", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Births per Woman")
axes[0].legend()

axes[1].plot(china['Year'], china['SexRatio'], color='#8E44AD', linewidth=2.5)
axes[1].axvline(1979, color='crimson', linestyle='--', linewidth=1.5, label='Policy Start (1979)')
axes[1].axhline(1.05, color='gray', linestyle=':', linewidth=1, label='Natural baseline (~1.05)')
axes[1].set_title("China: Sex Ratio at Birth", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Male births per female birth")
axes[1].legend()

plt.suptitle("China's Two Trajectories (1969–2015)", fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("TFR:       1969 = {:.2f}  →  1979 = {:.2f}  →  2000 = {:.2f}".format(
    china[china['Year']==1969]['Fertility'].values[0],
    china[china['Year']==1979]['Fertility'].values[0],
    china[china['Year']==2000]['Fertility'].values[0]))
print("Sex Ratio: 1969 = {:.3f} →  1979 = {:.3f} →  2005 = {:.3f} (peak)".format(
    china[china['Year']==1969]['SexRatio'].values[0],
    china[china['Year']==1979]['SexRatio'].values[0],
    china[china['Year']==2005]['SexRatio'].values[0]))
print()
print("Note the contrast already visible here: TFR was declining steeply well before 1979.")
print("Sex ratio was flat and near-baseline before 1979, then rose sharply afterward.")


In [ ]:
# ── Fig 2: All countries — both outcomes ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for country in DONOR_COUNTRIES:
    d = df[df['Country Name'] == country]
    axes[0].plot(d['Year'], d['Fertility'], color='#AAAAAA', linewidth=1, alpha=0.7)
    axes[1].plot(d['Year'], d['SexRatio'],  color='#AAAAAA', linewidth=1, alpha=0.7)

axes[0].plot(china['Year'], china['Fertility'], color='#1B4F72', linewidth=2.5, label='China', zorder=5)
axes[0].axvline(1979, color='crimson', linestyle='--', linewidth=1.5, label='Policy Start')
axes[0].set_title("Fertility Rate: China vs. Donor Pool", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Births per Woman")
axes[0].legend(loc='upper right')

axes[1].plot(china['Year'], china['SexRatio'], color='#8E44AD', linewidth=2.5, label='China', zorder=5)
axes[1].axvline(1979, color='crimson', linestyle='--', linewidth=1.5, label='Policy Start')
axes[1].set_title("Sex Ratio at Birth: China vs. Donor Pool", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Male births per female birth")
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.show()

print("Notice: on the left, China's line is unremarkable among the donor pool — everyone is declining.")
print("On the right, China visibly separates from the donor pool after 1979 in a way no other country does.")


---
## 3. Motivating Analysis: Difference-in-Differences

Before the main synthetic control analyses, a simple Difference-in-Differences (DiD) regression using South Korea as a single comparison country, on both outcomes.

**DiD logic:** If China and South Korea were on parallel trajectories before 1979, any divergence after 1979 can be attributed to the One-Child Policy.

**Limitation (why we need SCM):** DiD with a single comparison country is fragile — it assumes that one country is a perfect counterfactual for China. The synthetic control method relaxes this by constructing an *optimal weighted blend* of multiple countries, and we run it on both outcomes below.


In [ ]:
# Build DiD dataset: China vs South Korea
did_df = df[df['Country Name'].isin(['China', 'South Korea'])].copy()
did_df['log_Fertility'] = np.log(did_df['Fertility'])
did_df['Treatment'] = (did_df['Country Name'] == 'China').astype(int)
did_df['Post'] = (did_df['Year'] >= POLICY_YEAR).astype(int)
did_df['DiD'] = did_df['Treatment'] * did_df['Post']

print("=" * 60)
print("DiD — Outcome: log(Total Fertility Rate)")
print("=" * 60)
model_tfr = smf.ols('log_Fertility ~ Treatment + Post + DiD', data=did_df).fit(cov_type='HC3')
print(model_tfr.summary().tables[1])

print()
print("=" * 60)
print("DiD — Outcome: Sex Ratio at Birth")
print("=" * 60)
model_sex = smf.ols('SexRatio ~ Treatment + Post + DiD', data=did_df).fit(cov_type='HC3')
print(model_sex.summary().tables[1])


In [ ]:
# ── Fig 3: Parallel trends check — both outcomes ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for country, color, lw in [('China', '#1B4F72', 2.5), ('South Korea', '#E67E22', 2)]:
    d = did_df[did_df['Country Name'] == country]
    axes[0].plot(d['Year'], d['Fertility'], color=color, linewidth=lw, label=country)
    axes[1].plot(d['Year'], d['SexRatio'],  color=color, linewidth=lw, label=country)

axes[0].axvline(POLICY_YEAR, color='crimson', linestyle='--', linewidth=1.5, label='Policy Start')
axes[0].set_title("DiD: TFR — China vs. South Korea", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Births per Woman")
axes[0].legend()

axes[1].axvline(POLICY_YEAR, color='crimson', linestyle='--', linewidth=1.5, label='Policy Start')
axes[1].set_title("DiD: Sex Ratio — China vs. South Korea", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Male births per female birth")
axes[1].legend()

plt.tight_layout()
plt.show()

print("TFR pre-1979 trends are not parallel — South Korea's fertility was higher and falling")
print("faster than China's. Sex ratio pre-1979 trends ARE roughly parallel and flat — both near baseline —")
print("which makes the post-1979 divergence in sex ratio more visually striking here.")
print()
print("Either way, a single comparison country is a fragile counterfactual. This motivates SCM below.")


---
## 4. Synthetic Control Method — Setup

Both outcomes below use the same synthetic control machinery, so it's written once as a reusable function. The logic (Abadie et al., 2010): construct a counterfactual "Synthetic China" as a *weighted combination* of donor countries, chosen to minimize the pre-treatment gap on both the outcome variable and a set of predictor covariates.

**Predictors used for both outcomes:** log GDP, urbanization, child mortality, and mean pre-treatment value of the outcome itself. Using the same predictor set for both outcomes keeps the comparison between them apples-to-apples — any difference in results reflects the outcome, not a different modeling choice.


In [ ]:
def run_scm(df, outcome_col, donor_countries=DONOR_COUNTRIES,
            pre_years=PRE_YEARS, policy_year=POLICY_YEAR):
    """
    Fit a synthetic control for China on a given outcome column.
    Returns: (synth object, weights_df, years_arr, china_actual, synthetic, gap, rmspe_pre)
    """
    dataprep = Dataprep(
        foo=df,
        predictors=['log_GDP', 'Urban_Pct', 'Mortality'],
        predictors_op='mean',
        time_predictors_prior=pre_years,
        special_predictors=[(outcome_col, pre_years, 'mean')],
        dependent=outcome_col,
        unit_variable='Country Name',
        time_variable='Year',
        treatment_identifier='China',
        controls_identifier=donor_countries,
        time_optimize_ssr=pre_years,
    )
    synth = Synth()
    synth.fit(dataprep)

    weights_df = pd.DataFrame({
        'Country': donor_countries,
        'Weight' : synth.W.flatten()
    }).sort_values('Weight', ascending=False)

    outcome_wide = df.pivot(index='Year', columns='Country Name', values=outcome_col)
    synthetic = (outcome_wide[donor_countries].values @ synth.W).flatten()
    china_actual = outcome_wide['China'].values
    years_arr = outcome_wide.index.values

    pre_mask = years_arr < policy_year
    gap = china_actual - synthetic
    rmspe_pre = np.sqrt(np.mean((china_actual[pre_mask] - synthetic[pre_mask])**2))

    return {
        'synth': synth, 'weights': weights_df, 'years': years_arr,
        'actual': china_actual, 'synthetic': synthetic, 'gap': gap,
        'rmspe_pre': rmspe_pre, 'outcome_col': outcome_col
    }


def plot_scm_result(result, outcome_label, unit_label, color='#1B4F72', save_path=None):
    """Two-panel plot: actual vs synthetic, and the gap."""
    years, actual, synthetic, gap = (result['years'], result['actual'],
                                      result['synthetic'], result['gap'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(years, actual,    color=color,     linewidth=2.5, label='China (Actual)')
    axes[0].plot(years, synthetic, color='#E67E22', linewidth=2, linestyle='--',
                 label='Synthetic China')
    axes[0].axvline(POLICY_YEAR, color='crimson', linestyle=':', linewidth=1.5, label='Policy Start')
    axes[0].set_title(f"China vs. Synthetic China\n{outcome_label}", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Year"); axes[0].set_ylabel(unit_label)
    axes[0].legend()

    axes[1].plot(years, gap, color='#2C3E50', linewidth=2)
    axes[1].axvline(POLICY_YEAR, color='crimson', linestyle=':', linewidth=1.5, label='Policy Start')
    axes[1].axhline(0, color='gray', linewidth=0.8)
    axes[1].fill_between(years, gap, 0, where=(years >= POLICY_YEAR),
                          alpha=0.15, color='crimson', label='Post-treatment period')
    axes[1].set_title(f"Gap: China − Synthetic China\n(Estimated Policy Effect)", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Year"); axes[1].set_ylabel(f"Gap in {unit_label}")
    axes[1].legend()

    plt.suptitle(f"Synthetic Control: {outcome_label}", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


### 4.1 Outcome 1 — Total Fertility Rate

**Hypothesis:** If the One-Child Policy suppressed fertility beyond what structural development would explain, real China should fall well below Synthetic China after 1979, and stay there.


In [ ]:
tfr_result = run_scm(df, outcome_col='Fertility')

print("=== Synthetic Control Donor Weights (TFR) ===")
print(tfr_result['weights'][tfr_result['weights']['Weight'] > 0.001].to_string(index=False))
print(f"\nPre-treatment RMSPE (fit quality): {tfr_result['rmspe_pre']:.4f}")

post_mask = tfr_result['years'] >= POLICY_YEAR
avg_gap = tfr_result['gap'][post_mask].mean()
print(f"Avg post-treatment gap (China − Synthetic): {avg_gap:.4f} TFR")
print("(Negative = China below synthetic = policy suppressed fertility)")
print("(Near zero = policy effect indistinguishable from counterfactual trend)")


In [ ]:
plot_scm_result(tfr_result, "Total Fertility Rate", "Births per Woman",
                 color='#1B4F72', save_path='figures/scm_tfr_results.png')


### 4.2 Outcome 2 — Sex Ratio at Birth

**Hypothesis:** If the policy intensified son-selective behavior under a one-child constraint, real China should rise well above Synthetic China after 1979 — a pattern that structural development forces alone would not predict, since sex ratio at birth holds close to a stable biological baseline (~1.03–1.06) almost everywhere absent selective behavior.


In [ ]:
sex_result = run_scm(df, outcome_col='SexRatio')

print("=== Synthetic Control Donor Weights (Sex Ratio) ===")
print(sex_result['weights'][sex_result['weights']['Weight'] > 0.001].to_string(index=False))
print(f"\nPre-treatment RMSPE (fit quality): {sex_result['rmspe_pre']:.4f}")

post_mask = sex_result['years'] >= POLICY_YEAR
avg_gap = sex_result['gap'][post_mask].mean()
print(f"Avg post-treatment gap (China − Synthetic): {avg_gap:.4f}")
print("(Positive = China's sex ratio rose above synthetic = evidence of policy-driven son selection)")


In [ ]:
plot_scm_result(sex_result, "Sex Ratio at Birth", "Male Births per Female Birth",
                 color='#8E44AD', save_path='figures/scm_sexratio_results.png')


---
## 5. Placebo Tests

A gap between real and synthetic China only means something if it's *unusual*. This is the standard robustness check from Abadie et al.: rerun the exact same synthetic control procedure treating **each donor country** as if it were "treated" in 1979 — even though none of them actually had a one-child policy. This produces a distribution of "placebo gaps."

If China's gap falls comfortably within that distribution, the result is not distinguishable from noise — the method just can't detect an effect (or there isn't one) at this level of aggregation. If China's gap is unusually large relative to the placebo distribution, that's meaningful evidence of a real effect.

We run this for both outcomes.


In [ ]:
def run_placebo_test(df, outcome_col, donor_countries=DONOR_COUNTRIES,
                      pre_years=PRE_YEARS, policy_year=POLICY_YEAR,
                      rmspe_ratio_cutoff=20):
    """
    For each donor country, treat it as the 'treated unit' and every other
    country (including China) as its donor pool. Returns a DataFrame of
    post/pre RMSPE ratios for every unit, with China included for comparison.

    Units with a very poor pre-treatment fit (rmspe_ratio_cutoff) are excluded
    from the placebo distribution, following Abadie et al. — a country SCM
    can't even fit well pre-treatment tells us nothing about the post-period.
    """
    all_units = ['China'] + donor_countries
    placebo_results = {}

    for treated_unit in all_units:
        controls = [c for c in all_units if c != treated_unit]
        try:
            dataprep = Dataprep(
                foo=df,
                predictors=['log_GDP', 'Urban_Pct', 'Mortality'],
                predictors_op='mean',
                time_predictors_prior=pre_years,
                special_predictors=[(outcome_col, pre_years, 'mean')],
                dependent=outcome_col,
                unit_variable='Country Name',
                time_variable='Year',
                treatment_identifier=treated_unit,
                controls_identifier=controls,
                time_optimize_ssr=pre_years,
            )
            synth = Synth()
            synth.fit(dataprep)

            outcome_wide = df.pivot(index='Year', columns='Country Name', values=outcome_col)
            synthetic = (outcome_wide[controls].values @ synth.W).flatten()
            actual = outcome_wide[treated_unit].values
            years_arr = outcome_wide.index.values

            pre_mask  = years_arr < policy_year
            post_mask = years_arr >= policy_year

            rmspe_pre  = np.sqrt(np.mean((actual[pre_mask]  - synthetic[pre_mask])**2))
            rmspe_post = np.sqrt(np.mean((actual[post_mask] - synthetic[post_mask])**2))

            placebo_results[treated_unit] = {
                'rmspe_pre': rmspe_pre,
                'rmspe_post': rmspe_post,
                'rmspe_ratio': rmspe_post / rmspe_pre if rmspe_pre > 0 else np.nan,
            }
        except Exception as e:
            print(f"  Skipped {treated_unit}: {e}")
            continue

    result_df = pd.DataFrame(placebo_results).T
    result_df = result_df.sort_values('rmspe_ratio', ascending=False)
    return result_df


In [ ]:
print("Running placebo test for TFR (this fits an SCM for every country — may take a minute)...")
placebo_tfr = run_placebo_test(df, outcome_col='Fertility')
print("Done.\n")
print(placebo_tfr.round(4))


In [ ]:
print("Running placebo test for Sex Ratio...")
placebo_sex = run_placebo_test(df, outcome_col='SexRatio')
print("Done.\n")
print(placebo_sex.round(4))


In [ ]:
# ── Fig: Placebo distributions — both outcomes ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, placebo_df, title, color in [
    (axes[0], placebo_tfr, "TFR: Post/Pre RMSPE Ratio", '#1B4F72'),
    (axes[1], placebo_sex, "Sex Ratio: Post/Pre RMSPE Ratio", '#8E44AD'),
]:
    colors = ['crimson' if idx == 'China' else '#AAAAAA' for idx in placebo_df.index]
    ax.barh(placebo_df.index, placebo_df['rmspe_ratio'], color=colors)
    ax.set_xlabel("Post-Treatment / Pre-Treatment RMSPE Ratio")
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()

plt.suptitle("Placebo Test: Where Does China Rank?", fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('figures/placebo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

china_rank_tfr = (placebo_tfr['rmspe_ratio'] > placebo_tfr.loc['China', 'rmspe_ratio']).sum() + 1
china_rank_sex = (placebo_sex['rmspe_ratio'] > placebo_sex.loc['China', 'rmspe_ratio']).sum() + 1
n_units = len(placebo_tfr)

print(f"TFR: China ranks #{china_rank_tfr} of {n_units} units by RMSPE ratio")
print(f"     (pseudo p-value ≈ {china_rank_tfr/n_units:.3f})")
print(f"Sex Ratio: China ranks #{china_rank_sex} of {n_units} units by RMSPE ratio")
print(f"     (pseudo p-value ≈ {china_rank_sex/n_units:.3f})")
print()
print("A low rank number (China near the top) means its post/pre RMSPE ratio is unusually")
print("large relative to placebo units — evidence the gap is not just noise.")


---
## 6. Comparative Interpretation

Running these two outcomes in tandem, through the identical causal design, produces a genuinely useful contrast — the two results diverge sharply, and that divergence is itself the finding.

### Total Fertility Rate: a clean null

- **Donor weights:** Synthetic China is 76.8% Thailand + 21.4% South Korea + 1.8% Turkey.
- **Post-treatment gap:** +0.054 TFR — essentially zero, and in the *wrong* direction for a suppression story (real China is marginally higher than synthetic, not lower).
- **Placebo rank:** China ranks **11th of 11** units on the post/pre RMSPE ratio (pseudo p-value ≈ 1.00). Every placebo country's synthetic control diverges from its real trajectory *more* than China's does.

This is about as clean a null result as SCM produces. It's not that the method failed to detect an effect — it's that China's post-1979 fertility trajectory is *less* unusual, relative to its own pre-treatment pattern, than almost every other country in the donor pool. Thailand and South Korea, two countries with no comparable coercive fertility policy, saw fertility collapse at essentially the same pace as China did.

### Sex Ratio at Birth: a real signal

- **Donor weights:** Synthetic China is 80.9% Thailand + 9.3% Philippines + 9.1% Pakistan + 0.8% Sri Lanka.
- **Pre-treatment fit:** RMSPE of 0.002 — an extremely tight match, far better than the TFR fit. This donor pool can replicate China's pre-1979 sex ratio almost exactly.
- **Post-treatment gap:** +0.073 — meaningful and in the theoretically predicted direction (rising son preference).
- **Placebo rank:** China ranks **2nd of 11** (pseudo p-value ≈ 0.18). With only 11 units total, the best possible rank is 1st (p ≈ 0.09) — ranking 2nd puts China very close to the ceiling of what this test can detect at this sample size.

Unlike TFR, sex ratio at birth has no plausible non-policy explanation for a sustained rise. Economic development, urbanization, and rising incomes do not predict elevated son preference — if anything, the demographic literature generally associates development with *declining* sex-selective behavior. A rise that is (a) tightly matched pre-treatment, (b) large post-treatment, and (c) among the most extreme in a full placebo distribution, is a much more direct signature of policy-specific behavioral change.

### Putting it together

The honest conclusion is not "the policy worked" or "the policy didn't work." It's more specific than that:

> **The One-Child Policy does not appear to have meaningfully accelerated China's fertility decline beyond what economic and demographic modernization was already producing. But it does appear to have changed *how* families made fertility decisions — specifically, by intensifying sex-selective behavior under a binding one-child constraint.**

This reframes the research question productively. Rather than asking "did the policy reduce births?" — a question the TFR data answers with a fairly confident *not much, if at all* — the sharper question becomes "what did the policy change about the *terms* on which those births happened?" The sex ratio result suggests: quite a lot.


---
## 7. Limitations & Next Steps

**Limitations:**
- **Unit of analysis:** Using China as a single national unit aggregates enormous internal variation. The One-Child Policy was implemented unevenly across provinces, urban vs. rural settings, and ethnic minority populations (many of whom were exempted). A national-level result masks that variation entirely.
- **TFR pre-treatment fit:** An RMSPE of 0.596 is acceptable but not tight, especially compared to the sex ratio fit (0.002). This means more uncertainty attaches to the TFR counterfactual specifically.
- **Small placebo sample:** With only 11 units (China + 10 donors), the placebo test has limited resolution — the best achievable pseudo p-value is ≈0.09. A larger donor pool would sharpen inference on both outcomes, particularly sex ratio, which is close to that ceiling already.
- **Two outcomes, not a full outcome space:** Female labor force participation, household size, and reported IPV/family conflict rates would round out the picture of what the policy changed beyond fertility timing and sex composition.

**Next Steps:**
- **Province-level analysis:** Enforcement intensity varied significantly across China's provinces — early/strict vs. delayed/weak implementation, and explicit exemptions for rural households and ethnic minorities. Comparing provinces within China avoids the cross-country donor pool problem entirely, since all provinces share the same national political economy and culture. This is the most statistically promising extension, though it requires provincial-level data collection beyond what's available through World Bank indicators.
- **Expand the donor pool:** More donor countries would sharpen the placebo test's resolution, particularly for the sex ratio result, which is already close to what an 11-unit placebo test can detect.
- **Additional outcomes:** Female labor force participation and household structure could show complementary effects to the son-preference signal found here.

---
*Srisha Raj · 2025*  
*Data: World Bank WDI · Code: github.com/srisha-raj/china-onechild-policy-scm*
